# Desafio 04 — Serviços de Computação em Nuvem para Machine Learning
## Classificação do impacto das redes sociais na vida dos estudantes

**Objetivo:** evoluir o projeto da Sprint 03 para um fluxo documentado de ML em nuvem: armazenamento, preparação, treinamento, avaliação, empacotamento, inferência e monitoramento.

> **Importante:** as células de AWS dependem de uma conta AWS, credenciais temporárias válidas, bucket S3, papel IAM e permissões do SageMaker. Não execute a célula de implantação sem confirmar custos e permissões. A criação de endpoint pode gerar cobrança enquanto estiver ativo.

A base enviada contém 4.500 registros e 16 colunas. A variável-alvo é `Overall_Impact`.

## 1. Preparação do ambiente
Este notebook pode ser executado no Google Colab. Primeiro, faça upload do CSV quando solicitado.

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn joblib boto3 sagemaker

In [ ]:
import os, json, tarfile, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
TARGET = "Overall_Impact"
ID_COL = "Student_ID"
LABELS = ["Beneficial", "Neutral", "Negative"]

## 2. Carregamento da base
No Colab, selecione o arquivo `Social_media_impact_on_life.csv` enviado para esta atividade. Se estiver no mesmo diretório do notebook, a leitura será automática.

In [ ]:
CSV_PATH = "/content/Social_media_impact_on_life.csv"

if not os.path.exists(CSV_PATH):
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = next(name for name in uploaded if name.lower().endswith(".csv"))

df = pd.read_csv(CSV_PATH)
print("Dimensão:", df.shape)
display(df.head())

## 3. Auditoria e análise exploratória
A auditoria confirma dimensões, tipos, ausências, duplicatas e distribuição da variável-alvo. O conjunto tem forte desequilíbrio entre classes; por isso, além da acurácia, serão analisadas métricas macro e a matriz de confusão.

In [ ]:
print("Dimensão:", df.shape)
print("\nTipos:")
display(df.dtypes.rename("tipo").to_frame())
print("\nValores ausentes:")
display(df.isna().sum().rename("ausentes").to_frame())
print("\nLinhas duplicadas:", df.duplicated().sum())
print("IDs duplicados:", df[ID_COL].duplicated().sum())
print("\nDistribuição da variável-alvo:")
display(df[TARGET].value_counts().rename("quantidade").to_frame())
display((df[TARGET].value_counts(normalize=True)*100).round(2).rename("percentual").to_frame())

ax = df[TARGET].value_counts().reindex(LABELS).plot(kind="bar", rot=0)
ax.set_title("Distribuição das classes — Overall_Impact")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade")
plt.tight_layout()
plt.show()

## 4. Preparação dos dados e divisão estratificada
`Student_ID` é removido por ser identificador, não atributo preditivo. Os valores ausentes são tratados dentro dos pipelines, ajustados somente com o conjunto de treinamento, evitando vazamento de informação. Variáveis categóricas são codificadas com One-Hot Encoding. A Regressão Logística também utiliza padronização numérica.

In [ ]:
assert TARGET in df.columns, f"Coluna-alvo não encontrada: {TARGET}"
assert df[TARGET].notna().all(), "A variável-alvo possui valores ausentes."

X = df.drop(columns=[TARGET, ID_COL], errors="ignore")
y = df[TARGET].astype(str)

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
numeric_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer([
    ("num", numeric_lr, numeric_cols),
    ("cat", categorical, categorical_cols)
])
preprocessor_rf = ColumnTransformer([
    ("num", numeric_rf, numeric_cols),
    ("cat", categorical, categorical_cols)
])

print("Treino:", X_train.shape, "Teste:", X_test.shape)
print("Variáveis numéricas:", numeric_cols)
print("Variáveis categóricas:", categorical_cols)

## 5. Treinamento e comparação inicial
Os dois algoritmos do relatório anterior são mantidos: Regressão Logística e Random Forest. A comparação utiliza o mesmo particionamento estratificado.

In [ ]:
models = {
    "Regressão Logística": Pipeline([
        ("prep", preprocessor_lr),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor_rf),
        ("model", RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

comparison = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    p, r, f, _ = precision_recall_fscore_support(
        y_test, pred, average="macro", zero_division=0
    )
    comparison.append({
        "Modelo": name,
        "Acurácia": accuracy_score(y_test, pred),
        "Precisão macro": p,
        "Recall macro": r,
        "F1 macro": f
    })

comparison_df = pd.DataFrame(comparison).set_index("Modelo")
display(comparison_df.round(4))

## 6. Otimização da Regressão Logística
A busca em grade usa validação cruzada estratificada de cinco partições e F1 macro como critério, pois as classes são desbalanceadas. O conjunto de teste permanece reservado para avaliação final.

In [ ]:
param_grid = {
    "model__C": [0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=models["Regressão Logística"],
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Melhores parâmetros:", grid.best_params_)
print("Melhor F1 macro médio na validação cruzada:", round(grid.best_score_, 4))

## 7. Avaliação final no conjunto de teste
Registre os resultados efetivamente exibidos após executar o notebook. Não substitua métricas por valores estimados.

In [ ]:
y_pred = best_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

print(f"Acurácia: {acc:.4%}")
print(f"Precisão macro: {precision_macro:.4%}")
print(f"Recall macro: {recall_macro:.4%}")
print(f"F1 macro: {f1_macro:.4%}")
print("\nRelatório por classe:")
print(classification_report(y_test, y_pred, labels=LABELS, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(cmap="Blues", values_format="d")
plt.title("Matriz de confusão — modelo otimizado")
plt.tight_layout()
plt.show()

metrics = {
    "accuracy": float(acc),
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "best_params": grid.best_params_,
    "confusion_matrix_labels": LABELS,
    "confusion_matrix": cm.tolist()
}
with open("metricas_modelo.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

## 8. Teste local de inferência e versionamento do artefato
Antes de implantar na nuvem, o modelo é salvo e testado localmente. A versão do artefato e as métricas são registradas em um manifesto.

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

model_path = ARTIFACT_DIR / "model.joblib"
joblib.dump(best_model, model_path)

sample = X_test.iloc[[0]]
local_prediction = best_model.predict(sample)[0]
print("Classe real:", y_test.iloc[0])
print("Previsão local:", local_prediction)

manifest = {
    "model_name": "social-media-overall-impact",
    "model_version": "1.0.0",
    "target": TARGET,
    "features": X.columns.tolist(),
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "random_state": RANDOM_STATE,
    "metrics": metrics
}
with open(ARTIFACT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(model_path, arcname="model.joblib")

print("Artefatos criados:", list(ARTIFACT_DIR.iterdir()))

## 9. Armazenamento no Amazon S3
**Pré-requisitos:** credenciais AWS temporárias válidas e bucket S3 já criado na região escolhida. Não coloque chaves de acesso diretamente no código ou no notebook compartilhado. No Colab, prefira Secrets; no AWS Academy, use as credenciais temporárias fornecidas pelo laboratório.

Preencha somente o nome do bucket e a região. A célula envia o CSV, o modelo empacotado e o manifesto.

In [ ]:
# Exemplo: bucket criado previamente no console AWS
AWS_REGION = "us-east-1"
S3_BUCKET = ""  # preencher com o nome real do bucket
S3_PREFIX = "desafio04/social-media-impact"

if not S3_BUCKET:
    print("PREENCHA S3_BUCKET após criar o bucket no Amazon S3.")
else:
    import boto3
    s3 = boto3.client("s3", region_name=AWS_REGION)
    s3.upload_file(CSV_PATH, S3_BUCKET, f"{S3_PREFIX}/data/Social_media_impact_on_life.csv")
    s3.upload_file("model.tar.gz", S3_BUCKET, f"{S3_PREFIX}/models/model.tar.gz")
    s3.upload_file(str(ARTIFACT_DIR / "manifest.json"), S3_BUCKET, f"{S3_PREFIX}/models/manifest.json")
    print("Uploads concluídos no S3.")

## 10. Código de inferência para o SageMaker
Esta célula cria o arquivo `inference.py`, usado pelo contêiner de inferência do SageMaker Scikit-learn. A implantação só deve ser feita depois de configurar a role de execução e confirmar permissões e custos.

In [ ]:
inference_source = r'''
import os, json, joblib
import pandas as pd

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))

def input_fn(request_body, content_type):
    if content_type != "application/json":
        raise ValueError("Content-Type deve ser application/json")
    payload = json.loads(request_body)
    records = payload.get("instances", payload)
    return pd.DataFrame(records)

def predict_fn(input_data, model):
    predictions = model.predict(input_data).tolist()
    probabilities = model.predict_proba(input_data).tolist()
    return {"predictions": predictions, "probabilities": probabilities}

def output_fn(prediction, accept):
    return json.dumps(prediction), "application/json"
'''
with open("inference.py", "w", encoding="utf-8") as f:
    f.write(inference_source)
print("inference.py criado.")

## 11. Implantação opcional de endpoint no SageMaker
**Atenção a custos:** endpoint em tempo real gera cobrança enquanto estiver ativo. Use somente se sua conta/laboratório permitir. A role IAM precisa ter permissões para SageMaker, S3 e CloudWatch. Em ambientes AWS Academy, verifique os serviços permitidos e o limite de créditos.

O código abaixo é um roteiro para implantação; preencha `ROLE_ARN` e `MODEL_S3_URI` com valores reais. A versão do framework deve ser compatível com a região/conta.

In [ ]:
# DESCOMENTE E PREENCHA SOMENTE APÓS VALIDAR PERMISSÕES E CUSTOS.
#
# import sagemaker
# from sagemaker.sklearn.model import SKLearnModel
#
# ROLE_ARN = "arn:aws:iam::<ACCOUNT_ID>:role/<SAGEMAKER_EXECUTION_ROLE>"
# MODEL_S3_URI = f"s3://{S3_BUCKET}/{S3_PREFIX}/models/model.tar.gz"
# ENDPOINT_NAME = "desafio04-social-impact-endpoint"
#
# sklearn_model = SKLearnModel(
#     model_data=MODEL_S3_URI,
#     role=ROLE_ARN,
#     entry_point="inference.py",
#     framework_version="1.2-1",
#     py_version="py3"
# )
#
# predictor = sklearn_model.deploy(
#     initial_instance_count=1,
#     instance_type="ml.m5.large",
#     endpoint_name=ENDPOINT_NAME
# )
#
# # Exemplo de payload: use uma linha real de X_test
# payload = {"instances": X_test.iloc[[0]].to_dict(orient="records")}
# response = predictor.predict(payload, initial_args={"ContentType": "application/json"})
# print(response)

## 12. Monitoramento com Amazon CloudWatch
Após uma chamada real ao endpoint, o SageMaker publica métricas operacionais no namespace `AWS/SageMaker`. No console CloudWatch, consulte `Invocations`, `ModelLatency`, `OverheadLatency`, `Invocation4XXErrors` e `Invocation5XXErrors`, filtrando pelo endpoint e variante.

Registre capturas de tela do painel e do período consultado. Para monitorar qualidade preditiva (data drift/model drift), será necessário comparar dados de produção com dados de referência e, quando houver rótulos reais, acompanhar métricas de qualidade. Métricas de infraestrutura sozinhas não comprovam a qualidade do modelo.

## 13. Encerramento e limpeza de recursos
Exclua o endpoint depois dos testes para evitar cobranças. Não exclua o bucket se precisar preservar as evidências.

In [ ]:
# Execute somente se um endpoint tiver sido realmente criado:
#
# predictor.delete_endpoint()
# print("Endpoint removido.")

## 14. Evidências que devem ser anexadas ao relatório
1. Bucket S3 e objetos armazenados (CSV, artefato do modelo e manifesto).
2. Notebook executado com auditoria, treinamento e métricas.
3. Artefato `model.tar.gz` e manifesto de versão.
4. Endpoint SageMaker ativo e teste de inferência, caso implantado.
5. Métricas do endpoint no CloudWatch, caso disponíveis.
6. Registro de exclusão do endpoint após os testes, para evitar custos.

**Limitações:** a classificação descreve padrões presentes nesta base e não comprova causalidade. Como a classe `Negative` é minoritária, reporte métricas por classe e não utilize somente acurácia. A base contém indicadores autorrelatados de saúde/estresse; o modelo não deve ser usado para diagnóstico ou decisão individual de alto impacto.